# ProjectMem Colab Full Runner

Notebook nay goi `app/main.py` truc tiep theo full mode va full benchmark suite.

- `USE_DUMMY = False`: bat che do model that
- `MAX_SCENARIOS = None`: chay full dataset cua tung benchmark
- `INCLUDE_LOCAL_MEMAE = True` chi khi ban da co file parquet local cho `memae`


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Dung205789/MultiAgentConflictResolution"
BRANCH = "main"
WORKDIR = "/content/ProjectMem"

if os.path.isdir(WORKDIR):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", WORKDIR, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", WORKDIR, "pull", "origin", BRANCH], check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("Working directory:", os.getcwd())
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())


In [ ]:
%cd /content/ProjectMem
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN not set. Public benchmarks and public Qwen variants can still download.")


In [ ]:
import torch

USE_DUMMY = False
MAX_SCENARIOS = None
DEVICE = "auto"
OUTPUT_ROOT = "reports/colab_full_qwen_8b_3b"

MODEL_NAME1 = "Qwen/Qwen3-8B"
MODEL_NAME2 = "Qwen/Qwen2.5-3B-Instruct"

INCLUDE_LOCAL_MEMAE = False
MEMAE_LOCAL_PATH = "data/raw/memab/Conflict_Resolution-00000-of-00001.parquet"

BENCHMARK_SPECS = [
    {"benchmark": "mab_conflict"},
    {"benchmark": "mab", "subset": "all"},
    {"benchmark": "longmemeval", "subset": "oracle"},
    {"benchmark": "safeflow"},
    {"benchmark": "locomo", "subset": "all"},
]

if INCLUDE_LOCAL_MEMAE:
    BENCHMARK_SPECS.insert(0, {"benchmark": "memae"})

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Agent 1 model:", MODEL_NAME1)
print("Agent 2 model:", MODEL_NAME2)
print("Benchmark specs:", BENCHMARK_SPECS)
print("Include local memae:", INCLUDE_LOCAL_MEMAE)


In [ ]:
import json
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

summary = {
    "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "benchmarks": [],
    "use_dummy": USE_DUMMY,
    "device": DEVICE,
    "agent1_model": None if USE_DUMMY else MODEL_NAME1,
    "agent2_model": None if USE_DUMMY else MODEL_NAME2,
    "max_scenarios": MAX_SCENARIOS,
}

if INCLUDE_LOCAL_MEMAE and not Path(MEMAE_LOCAL_PATH).exists():
    print(f"WARNING: {MEMAE_LOCAL_PATH} not found. memae run will fail unless you upload that parquet first.")

for spec in BENCHMARK_SPECS:
    benchmark = spec["benchmark"]
    run_name = benchmark if "subset" not in spec else f"{benchmark}_{spec['subset']}"
    output_dir = output_root / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "app/main.py",
        "--benchmark", benchmark,
        "--output-dir", str(output_dir),
        "--device", DEVICE,
    ]

    if MAX_SCENARIOS is not None:
        cmd.extend(["--max-scenarios", str(MAX_SCENARIOS)])

    if "subset" in spec:
        cmd.extend(["--subset", spec["subset"]])

    if USE_DUMMY:
        cmd.append("--use-dummy")
    else:
        cmd.extend([
            "--agent1-model", MODEL_NAME1,
            "--agent2-model", MODEL_NAME2,
        ])

    print("\n" + "=" * 72)
    print(f"Running full benchmark via app/main.py: {run_name}")
    print(shlex.join(cmd))
    print("=" * 72)

    completed = subprocess.run(cmd, check=False)

    summary["benchmarks"].append({
        "benchmark": benchmark,
        "run_name": run_name,
        "subset": spec.get("subset"),
        "returncode": completed.returncode,
        "status": "success" if completed.returncode == 0 else "failed",
        "output_dir": str(output_dir),
    })

summary_path = output_root / "matrix_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved summary:", summary_path)


In [ ]:
import zipfile
from pathlib import Path

output_root = Path(OUTPUT_ROOT)
for path in sorted(output_root.rglob("*.json")):
    print(path)

zip_path = Path("/content/projectmem_full_reports.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_root.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(output_root.parent))

print("Created:", zip_path)
